<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/20-llms-in-practice/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 20: Exercises

Twelve exercises: four Easy, five Medium and three Hard. They're the building blocks of real LLM applications, and none of them needs a model download, so they run in seconds on any computer. The Easy ones are search and prompt basics: cosine similarity, top-k, chunking text, and the ChatML format. The Medium ones build and measure a RAG system: the prompt with a size budget, retrieval metrics, answer scoring, pulling JSON out of a reply, and nucleus (top-p) sampling. The Hard ones are three techniques behind modern LLM tools: BM25 keyword search, a LoRA layer, and 8-bit quantisation.

**Run the setup cell first.**

Rules:

- Python, NumPy and PyTorch. Don't call a library that does the whole job for you (such as scikit-learn's `cosine_similarity` or the `peft` library); the point is to see how each piece works.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import collections
import json
import math
import string

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):
            return False
        a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, torch.Tensor):
        body = np.array2string(x.detach().cpu().numpy(), precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"tensor(shape={tuple(x.shape)}, dtype={str(x.dtype).replace('torch.', '')}, {' '.join(body.split())})"
    elif isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, (np.ndarray, torch.Tensor)) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_cosine_similarity_matrix(f):
    rng = np.random.default_rng(0)
    A, B = rng.normal(size=(4, 6)), rng.normal(size=(3, 6))
    expected = (A / np.linalg.norm(A, axis=1, keepdims=True)) @ (B / np.linalg.norm(B, axis=1, keepdims=True)).T
    with_zero = np.vstack([A[:2], np.zeros((1, 6))])
    _scenarios("cosine_similarity_matrix", [
        ("the example", lambda: f(np.array([[1.0, 0.0]]), np.array([[1.0, 1.0], [0.0, 2.0]])), np.array([[2 ** -0.5, 0.0]]), None),
        ("random vectors", lambda: f(A, B), expected, "divide each row by its own length"),
        ("lengths don't matter", lambda: f(A * 10, B * 0.1), expected, None),
        ("a row of zeros gives 0, not nan", lambda: f(with_zero, B)[2], np.zeros(3), "skip the division where the length is 0"),
        ("shape", lambda: f(A, B).shape, (4, 3), None),
    ])


def check_top_k_indices(f):
    rng = np.random.default_rng(1)
    scores = rng.random(50)
    ties = np.random.default_rng(5).choice([0.1, 0.2, 0.3], 300)
    _scenarios("top_k_indices", [
        ("the example", lambda: f([0.2, 0.9, 0.5, 0.9], 3), [1, 3, 2], "ties: the earlier position first"),
        ("top 5 of 50", lambda: f(scores, 5), [int(i) for i in np.argsort(-scores)[:5]], None),
        ("all equal", lambda: f([0.5, 0.5, 0.5, 0.5], 4), [0, 1, 2, 3], "use a stable sort"),
        ("many ties", lambda: f(ties, 40), [int(i) for i in np.argsort(-ties, kind="stable")[:40]], "use a stable sort"),
        ("k larger than the list", lambda: f([3.0, 1.0], 5), [0, 1], None),
        ("returns Python ints", lambda: sorted({type(i).__name__ for i in f(scores, 3)}), ["int"], None),
    ])


def check_chunk_words(f):
    words = " ".join(f"w{i}" for i in range(10))
    _scenarios("chunk_words", [
        ("the example", lambda: f("a b c d e f g", 3, 1), ["a b c", "c d e", "e f g"], None),
        ("no overlap", lambda: f(words, 4, 0), ["w0 w1 w2 w3", "w4 w5 w6 w7", "w8 w9"], None),
        ("overlap 2, a short last chunk", lambda: f(words, 4, 2), ["w0 w1 w2 w3", "w2 w3 w4 w5", "w4 w5 w6 w7", "w6 w7 w8 w9"],
         "stop once a chunk reaches the last word"),
        ("text shorter than one chunk", lambda: f("just three words", 5, 2), ["just three words"], None),
        ("extra whitespace", lambda: f("  a\n b\t c  ", 2, 0), ["a b", "c"], None),
        ("empty text", lambda: f("", 3, 1), [], None),
        ("overlap equal to size raises", lambda: f(words, 3, 3), Raises(ValueError), None),
        ("overlap larger than size raises", lambda: f(words, 3, 5), Raises(ValueError), "check 0 <= overlap < size"),
        ("negative overlap raises", lambda: f(words, 3, -1), Raises(ValueError), None),
    ])


def check_format_chatml(f):
    convo = [{"role": "system", "content": "Be brief."}, {"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello!"},
             {"role": "user", "content": "Two\nlines"}]
    expected = ("<|im_start|>system\nBe brief.<|im_end|>\n<|im_start|>user\nHi<|im_end|>\n"
                "<|im_start|>assistant\nHello!<|im_end|>\n<|im_start|>user\nTwo\nlines<|im_end|>\n")
    _scenarios("format_chatml", [
        ("the example", lambda: f([{"role": "user", "content": "Hi"}], add_generation_prompt=True),
         "<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\n", None),
        ("a whole conversation", lambda: f(convo), expected, "role, new line, content, <|im_end|>, new line"),
        ("with the generation prompt", lambda: f(convo, add_generation_prompt=True), expected + "<|im_start|>assistant\n", None),
        ("no messages", lambda: f([]), "", None),
        ("an unknown role raises", lambda: f([{"role": "assistent", "content": "x"}]), Raises(ValueError), None),
    ])


def check_build_rag_prompt(f):
    head = "Answer the question using only the sources below. If they don't contain the answer, say \"I don't know\".\n\nSources:\n"
    chunks = ["Cards cost 2 pounds.", "Open 9 to 7.", "Wi-Fi is free."]
    two = len("[1] Cards cost 2 pounds.\n[2] Open 9 to 7.")

    def full(sources, q="How much is a card?"):
        return head + sources + f"\n\nQuestion: {q}\nAnswer:"

    _scenarios("build_rag_prompt", [
        ("three sources", lambda: f("How much is a card?", chunks), full("[1] Cards cost 2 pounds.\n[2] Open 9 to 7.\n[3] Wi-Fi is free."),
         "check every character of the template"),
        ("a limit that fits two", lambda: f("How much is a card?", chunks, max_chars=two), full("[1] Cards cost 2 pounds.\n[2] Open 9 to 7."),
         "the sources block, with its new lines, must be at most max_chars"),
        ("a limit exactly one character short", lambda: f("How much is a card?", chunks, max_chars=two - 1), full("[1] Cards cost 2 pounds."), None),
        ("stops at the first that doesn't fit", lambda: f("q", ["short", "a much longer chunk of text", "tiny"], max_chars=20),
         full("[1] short", q="q"), None),
        ("nothing fits", lambda: f("q", chunks, max_chars=5), full("(no sources)", q="q"), None),
        ("no chunks", lambda: f("q", []), full("(no sources)", q="q"), None),
    ])


def check_retrieval_metrics(f):
    results = [["a", "b", "c", "d"], ["e", "f", "g", "h"], ["i", "j", "k", "l"]]
    relevant = [{"a", "c"}, {"h"}, {"z"}]
    _scenarios("retrieval_metrics", [
        ("the example", lambda: f([["a", "b", "c"], ["d", "e", "f"]], [{"b"}, {"x"}], 2), {"recall@k": 0.5, "mrr": 0.25}, None),
        ("two relevant documents", lambda: f(results[:1], relevant[:1], 2), {"recall@k": 0.5, "mrr": 1.0},
         "recall divides by the number of relevant documents"),
        ("the first hit beyond k still counts for MRR", lambda: f(results[1:2], relevant[1:2], 2), {"recall@k": 0.0, "mrr": 0.25}, None),
        ("three queries", lambda: f(results, relevant, 3), {"recall@k": (1.0 + 0 + 0) / 3, "mrr": (1 + 0.25 + 0) / 3}, None),
        ("returns floats", lambda: sorted({type(v).__name__ for v in f(results, relevant, 3).values()}), ["float"], None),
        ("mismatched lengths raise", lambda: f(results, relevant[:2], 3), Raises(ValueError), None),
    ])


def check_answer_f1(f):
    _scenarios("answer_f1", [
        ("the example", lambda: f("It costs 2 pounds.", "2 pounds"), 2 / 3, None),
        ("identical after normalising", lambda: f("The library card!", "library card"), 1.0, "lower case, remove punctuation and a/an/the"),
        ("no words in common", lambda: f("Saturday", "10:00"), 0.0, None),
        ("repeated words count once", lambda: f("no no no", "no"), 0.5, "use Counter(...) & Counter(...)"),
        ("repeats on both sides", lambda: f("no no", "no no maybe"), 0.8, "a word repeated in both counts as often as in the shorter"),
        ("partial overlap", lambda: f("open on Saturday mornings", "Saturday from 10 to 4"), 2 * (1 / 4) * (1 / 5) / (1 / 4 + 1 / 5), None),
        ("both empty after normalising", lambda: f("The.", "a"), 1.0, None),
        ("one empty", lambda: f("", "2 pounds"), 0.0, None),
    ])


def check_extract_json(f):
    _scenarios("extract_json", [
        ("plain JSON", lambda: f('{"name": "Ana", "days": 7}'), {"name": "Ana", "days": 7}, None),
        ("the example, with a code fence and nesting",
         lambda: f('Sure! Here it is:\n```json\n{"name": "Ana", "tags": {"a": 1}}\n```'), {"name": "Ana", "tags": {"a": 1}},
         "count braces to find the matching }"),
        ("braces inside a string", lambda: f('Result: {"note": "smile :} or {", "ok": true} hope that helps'),
         {"note": "smile :} or {", "ok": True}, "ignore braces inside strings"),
        ("an escaped quote in a string", lambda: f('x {"q": "he said \\"hi}\\"", "n": 1} y'), {"q": 'he said "hi}"', "n": 1}, None),
        ("a false start before the real object", lambda: f('Use the {name} field. {"name": "Lee"}'), {"name": "Lee"},
         "if the first candidate doesn't parse, try the next {"),
        ("a list is not an object", lambda: f("[1, 2, 3]"), Raises(ValueError), None),
        ("no JSON at all", lambda: f("I'm sorry, I can't help with that."), Raises(ValueError), None),
    ])


def check_top_p_filter(f):
    g = torch.Generator().manual_seed(2)
    batch = torch.softmax(torch.randn(3, 12, generator=g, dtype=torch.float64) * 2, dim=-1)

    def reference(probs, p):
        out = torch.zeros_like(probs)
        for r in range(probs.shape[0]):
            order = sorted(range(probs.shape[1]), key=lambda i: (-probs[r, i].item(), i))
            total = 0.0
            for i in order:
                out[r, i] = probs[r, i]
                total += probs[r, i].item()
                if total >= p:
                    break
        return out / out.sum(dim=-1, keepdim=True)

    _scenarios("top_p_filter", [
        ("the example", lambda: f(torch.tensor([0.1, 0.5, 0.3, 0.1]), 0.7), torch.tensor([0.0, 0.625, 0.375, 0.0]), None),
        ("keeps the token that crosses p", lambda: f(torch.tensor([0.5, 0.3, 0.2]), 0.6), torch.tensor([0.625, 0.375, 0.0]),
         "keep while the total of the tokens before this one is below p"),
        ("the top token alone is enough", lambda: f(torch.tensor([0.05, 0.9, 0.05]), 0.5), torch.tensor([0.0, 1.0, 0.0]), None),
        ("a batch of rows", lambda: f(batch, 0.8), reference(batch, 0.8), None),
        ("ties keep the earlier token", lambda: f(torch.tensor([0.25, 0.25, 0.25, 0.25]), 0.4), torch.tensor([0.5, 0.5, 0.0, 0.0]),
         "use a stable sort"),
        ("p = 1 keeps everything", lambda: f(batch, 1.0), batch, None),
        ("exactly reaching p is enough", lambda: f(torch.tensor([0.5, 0.3, 0.2]), 0.5), torch.tensor([1.0, 0.0, 0.0]), None),
        ("p = 0 keeps only the top token", lambda: f(torch.tensor([0.2, 0.5, 0.3]), 0.0), torch.tensor([0.0, 1.0, 0.0]), "always keep the most likely token"),
        ("many ties keep the earliest", lambda: f(torch.full((64,), 1 / 64), 0.1), torch.cat([torch.full((7,), 1 / 7), torch.zeros(57)]), "use a stable sort"),
    ])


def _reference_bm25(query, docs, k1=1.5, b=0.75):
    n = len(docs)
    avgdl = sum(len(d) for d in docs) / n
    out = []
    for d in docs:
        s = 0.0
        for q in query:
            nq = sum(1 for dd in docs if q in dd)
            idf = math.log((n - nq + 0.5) / (nq + 0.5) + 1)
            tf = d.count(q)
            s += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(d) / avgdl))
        out.append(s)
    return np.array(out)


def check_bm25_scores(f):
    docs = [d.lower().split() for d in [
        "loans can be renewed up to three times online or by phone",
        "a lost card can be replaced at any desk for two pounds",
        "printing costs ten pence per page in black and white",
        "renewed loans renewed again renewed renewed",
        "the library card is free card card",
    ]]
    _scenarios("bm25_scores", [
        ("a one-word query", lambda: f(["renewed"], docs), _reference_bm25(["renewed"], docs), None),
        ("a longer query", lambda: f("lost library card".split(), docs), _reference_bm25("lost library card".split(), docs),
         "n(q) counts documents containing the word, not occurrences"),
        ("a repeated query word counts twice", lambda: f(["card", "card"], docs), 2 * _reference_bm25(["card"], docs), None),
        ("a word in no document", lambda: f(["swimming"], docs), np.zeros(5), None),
        ("k1 = 0.5, b = 0.3", lambda: f("renewed loans phone".split(), docs, k1=0.5, b=0.3),
         _reference_bm25("renewed loans phone".split(), docs, 0.5, 0.3), None),
        ("b = 0 ignores document length", lambda: f(["card"], [["card"], ["card"] + ["x"] * 20, ["y"]], b=0.0),
         _reference_bm25(["card"], [["card"], ["card"] + ["x"] * 20, ["y"]], b=0.0), None),
    ])


def check_LoRALinear(cls):
    x = torch.randn(5, 6, generator=torch.Generator().manual_seed(3))
    A_known = torch.arange(12, dtype=torch.float32).reshape(2, 6) / 10
    B_known = torch.arange(8, dtype=torch.float32).reshape(4, 2) / 10 - 0.3

    def base_layer(bias=True):
        torch.manual_seed(0)
        return nn.Linear(6, 4, bias=bias)

    def make(bias=True):
        return cls(base_layer(bias), r=2, alpha=4)

    def expected(bias=True, with_lora=True):
        base = base_layer(bias)
        with torch.no_grad():
            return base(x) + (2.0 * (x @ A_known.T @ B_known.T) if with_lora else 0)

    def with_known(layer):
        with torch.no_grad():
            layer.lora_A.copy_(A_known)
            layer.lora_B.copy_(B_known)
        return layer

    def output(bias=True):
        with torch.no_grad():
            return with_known(make(bias))(x)

    def merged_output(bias=True):
        with torch.no_grad():
            return with_known(make(bias)).merged()(x)

    def grads_only_lora():
        layer = make()
        with torch.no_grad():
            layer.lora_B.fill_(0.1)
        layer(x).sum().backward()
        return [layer.base.weight.grad is None, layer.base.bias.grad is None, layer.lora_A.grad is not None, layer.lora_B.grad is not None]

    def merge_is_separate():
        layer = with_known(make())
        before = layer.base.weight.detach().clone()
        merged = layer.merged()
        return [type(merged).__name__, merged is not layer.base, torch.equal(layer.base.weight, before)]

    def starts():
        with torch.no_grad():
            return make()(x)

    _scenarios("LoRALinear", [
        ("starts out identical to the base layer", starts, expected(with_lora=False), "B starts at zero"),
        ("shapes of A and B", lambda: [tuple(make().lora_A.shape), tuple(make().lora_B.shape)], [(2, 6), (4, 2)], None),
        ("only A and B are trainable", lambda: sum(p.numel() for p in make().parameters() if p.requires_grad), 2 * 6 + 4 * 2,
         "freeze the base layer's weight and bias"),
        ("gradients reach only A and B", grads_only_lora, [True, True, True, True], None),
        ("A is random, not zero", lambda: bool(make().lora_A.abs().sum() > 0), True, "initialise A with kaiming_uniform_"),
        ("output with known A and B", output, expected(), "scale the correction by alpha / r"),
        ("a base layer without bias", lambda: output(False), expected(False), None),
        ("the merged layer gives the same output", merged_output, expected(), "W + scaling * B @ A"),
        ("merging without bias", lambda: merged_output(False), expected(False), None),
        ("merged() returns a new nn.Linear and leaves the base alone", merge_is_separate, ["Linear", True, True], None),
    ])


def check_quantize_int8(f):
    g = torch.Generator().manual_seed(4)
    w = torch.randn(16, 64, generator=g) * torch.linspace(0.01, 3, 16).unsqueeze(1)

    def max_error_ratio():
        q, s = f(w)
        return bool(((q.float() * s - w).abs() <= s / 2 + 1e-6).all())

    def edge():
        row = torch.tensor([[1.0, -1.0, 0.5, 1.0 - 1e-7]])
        q, _ = f(row)
        return q[0].tolist()

    _scenarios("quantize_int8", [
        ("the example", lambda: [f(torch.tensor([[0.5, -1.27, 0.0]]))[0][0].tolist(), f(torch.tensor([[0.5, -1.27, 0.0]]))[1].tolist()],
         [[50, -127, 0], [[0.01]]], None),
        ("dtypes and shapes", lambda: [str(f(w)[0].dtype), tuple(f(w)[0].shape), str(f(w)[1].dtype), tuple(f(w)[1].shape)],
         ["torch.int8", (16, 64), "torch.float32", (16, 1)], "one scale per row, as a column"),
        ("one scale per row", lambda: f(w)[1], w.abs().amax(dim=1, keepdim=True) / 127, "scale = max(|row|) / 127"),
        ("every value within half a step", max_error_ratio, True, None),
        ("the largest value in each row becomes 127 or -127", lambda: f(w)[0].abs().amax(dim=1), torch.full((16,), 127, dtype=torch.int8), None),
        ("rounding to the nearest step", edge, [127, -127, 64, 127], None),
        ("a row of zeros", lambda: [f(torch.zeros(2, 3))[0].tolist(), f(torch.zeros(2, 3))[1].tolist()], [[[0, 0, 0], [0, 0, 0]], [[1.0], [1.0]]], None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `cosine_similarity_matrix`

Write `cosine_similarity_matrix(A, B)` that returns the cosine similarity between every row of `A` and every row of `B`.

- `A` has shape `(n, d)` and `B` has shape `(m, d)`; return a NumPy array of shape `(n, m)`.
- Entry `[i, j]` is the dot product of `A[i]` and `B[j]` divided by the product of their lengths.
- A row of all zeros has no direction: its similarity with anything is `0`, not `nan`.

Example: `cosine_similarity_matrix(np.array([[1.0, 0.0]]), np.array([[1.0, 1.0], [0.0, 2.0]]))` is about `[[0.707, 0.0]]`.

In [2]:
def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    # your code here
    pass

In [3]:
check_cosine_similarity_matrix(cosine_similarity_matrix)

FAIL  cosine_similarity_matrix: the example
      expected: array(shape=(1, 2), dtype=float64, [[0.7071, 0. ]])
      got:      None


### Exercise 2: `top_k_indices`

Write `top_k_indices(scores, k)` that returns the positions of the `k` highest scores, best first.

- `scores` is a 1D list or NumPy array.
- Return a plain Python list of `int`.
- When two scores are equal, the one that comes **first** in `scores` comes first in the result.
- If `k` is larger than the number of scores, return all of them.

Example: `top_k_indices([0.2, 0.9, 0.5, 0.9], 3)` is `[1, 3, 2]`.

In [4]:
def top_k_indices(scores, k: int) -> list:
    # your code here
    pass

In [5]:
check_top_k_indices(top_k_indices)

FAIL  top_k_indices: the example
      expected: [1, 3, 2]
      got:      None
      hint: ties: the earlier position first


### Exercise 3: `chunk_words`

Write `chunk_words(text, size, overlap)` that splits a long text into chunks of words for a RAG system, with some overlap between neighbouring chunks.

- Split the text into words on whitespace (`text.split()`).
- Each chunk has `size` words (the last one may have fewer) and starts `size - overlap` words after the previous one.
- Stop once a chunk reaches the last word, so no chunk is just a tail of the previous one.
- Return a list of strings, with the words of each chunk joined by single spaces. An empty text gives an empty list.
- Raise a `ValueError` if `size` is less than 1 or `overlap` isn't between 0 and `size - 1`.

Example: for the words `a b c d e f g` with `size=3` and `overlap=1`, the chunks are `"a b c"`, `"c d e"` and `"e f g"`.

In [6]:
def chunk_words(text: str, size: int, overlap: int) -> list:
    # your code here
    pass

In [7]:
check_chunk_words(chunk_words)

FAIL  chunk_words: the example
      expected: ['a b c', 'c d e', 'e f g']
      got:      None


### Exercise 4: `format_chatml`

Write `format_chatml(messages, add_generation_prompt=False)` that turns a conversation into the text format used by SmolLM2 and many other models (called ChatML).

- `messages` is a list of dictionaries with keys `"role"` and `"content"`.
- Each message becomes `<|im_start|>` + role + a new line + content + `<|im_end|>` + a new line.
- If `add_generation_prompt` is true, finish with `<|im_start|>assistant` and a new line, so the model knows it's its turn.
- Raise a `ValueError` if a role isn't `"system"`, `"user"` or `"assistant"`.

Example: `format_chatml([{"role": "user", "content": "Hi"}], add_generation_prompt=True)` is `"<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\n"`.

In [8]:
def format_chatml(messages, add_generation_prompt: bool = False) -> str:
    # your code here
    pass

In [9]:
check_format_chatml(format_chatml)

FAIL  format_chatml: the example
      expected: '<|im_start|>user\nHi<|im_end|>\n<|im_start|>assistant\n'
      got:      None


## Medium

### Exercise 5: `build_rag_prompt`

Write `build_rag_prompt(question, chunks, max_chars=None)` that builds the prompt for a RAG system, with numbered sources and a size limit.

The result must be exactly:

```
Answer the question using only the sources below. If they don't contain the answer, say "I don't know".

Sources:
[1] first chunk
[2] second chunk

Question: the question
Answer:
```

- Number the chunks from 1, one per line, in the order given.
- If `max_chars` is given, the sources block (the `[1] ...` lines joined with new lines) must be at most `max_chars` characters long: add chunks in order, and stop at the first one that would make it too long.
- If no chunk fits (or there are none), the sources block is `(no sources)`.

In [10]:
def build_rag_prompt(question: str, chunks, max_chars=None) -> str:
    # your code here
    pass

In [11]:
check_build_rag_prompt(build_rag_prompt)

FAIL  build_rag_prompt: three sources
      expected: 'Answer the question using only the sources below. If they don\'t contain the answer, s...
      got:      None
      hint: check every character of the template


### Exercise 6: `retrieval_metrics`

Write `retrieval_metrics(results, relevant, k)` that measures a search system over a set of test queries.

- `results` is a list with one ranked list of document ids per query (best first).
- `relevant` is a list with one set of relevant document ids per query.
- **recall@k** for one query is the number of relevant ids among the first `k` results, divided by the number of relevant ids.
- **reciprocal rank** for one query is `1 / position` of the first relevant id in the results (positions count from 1), or 0 if none appears at all.
- Return a dictionary `{"recall@k": ..., "mrr": ...}` with both averaged over the queries, as floats.
- Raise a `ValueError` if `results` and `relevant` have different lengths.

Example: for `results=[["a", "b", "c"], ["d", "e", "f"]]`, `relevant=[{"b"}, {"x"}]` and `k=2`, recall@2 is `(1 + 0) / 2 = 0.5` and MRR is `(1/2 + 0) / 2 = 0.25`.

In [12]:
def retrieval_metrics(results, relevant, k: int) -> dict:
    # your code here
    pass

In [13]:
check_retrieval_metrics(retrieval_metrics)

FAIL  retrieval_metrics: the example
      expected: {'recall@k': 0.5, 'mrr': 0.25}
      got:      None


### Exercise 7: `answer_f1`

Write `answer_f1(prediction, truth)` that scores a short answer against the correct one, the way question-answering benchmarks such as SQuAD do.

1. **Normalise** both strings: lower case, remove punctuation (every character in `string.punctuation`), remove the words `a`, `an` and `the`, and split into words.
2. Count the words they have in common, taking repeats into account (if a word appears twice in one and once in the other, it counts once).
3. Precision is common words / words in the prediction, recall is common words / words in the truth, and F1 is `2 * precision * recall / (precision + recall)`.
4. If both are empty after normalising, return `1.0`; if only one is, or nothing is in common, return `0.0`.

Example: `answer_f1("It costs 2 pounds.", "2 pounds")` is about `0.667` (2 common words; precision 2/4, recall 2/2).

In [14]:
def answer_f1(prediction: str, truth: str) -> float:
    # your code here
    pass

In [15]:
check_answer_f1(answer_f1)

FAIL  answer_f1: the example
      expected: 0.6666666666666666
      got:      None


### Exercise 8: `extract_json`

Write `extract_json(text)` that pulls a JSON object out of a language model's reply, which often has extra text around it.

- If the whole text (after removing surrounding whitespace) is valid JSON for an object, return it.
- Otherwise, find the first `{` and the `}` that **matches** it, counting nested braces, and ignoring braces inside JSON strings (between double quotes, taking `\"` escapes into account). Parse that part.
- Return a Python `dict`. Raise a `ValueError` if there's no valid JSON object.

Example: `extract_json('Sure! Here it is:\n```json\n{"name": "Ana", "tags": {"a": 1}}\n```')` is `{"name": "Ana", "tags": {"a": 1}}`.

(The lesson's simpler version, a regular expression for the first `{...}`, stops at the first `}`, so it fails on nested objects.)

In [16]:
def extract_json(text: str) -> dict:
    # your code here
    pass

In [17]:
check_extract_json(extract_json)

FAIL  extract_json: plain JSON
      expected: {'name': 'Ana', 'days': 7}
      got:      None


### Exercise 9: `top_p_filter`

Write `top_p_filter(probs, p)`, the **nucleus sampling** step used by `generate(top_p=...)`.

- `probs` is a tensor of probabilities over the last dimension (a single vector, or a batch of rows).
- In each row, keep the smallest set of the most likely tokens whose probabilities add up to **at least** `p`. Set the others to 0 and renormalise the kept ones so the row adds up to 1.
- Always keep at least the most likely token.
- Sort from most to least likely with a **stable** sort, so that among equal probabilities the earlier token is kept first.

Example: for `probs = torch.tensor([0.1, 0.5, 0.3, 0.1])` and `p = 0.7`, tokens 1 and 2 are kept (0.5 + 0.3 = 0.8 >= 0.7), giving `[0, 0.625, 0.375, 0]`.

In [18]:
def top_p_filter(probs: torch.Tensor, p: float) -> torch.Tensor:
    # your code here
    pass

In [19]:
check_top_p_filter(top_p_filter)

FAIL  top_p_filter: the example
      expected: tensor(shape=(4,), dtype=float32, [0. , 0.625, 0.375, 0. ])
      got:      None


## Hard

### Exercise 10: `bm25_scores`

Write `bm25_scores(query, documents, k1=1.5, b=0.75)`, the BM25 ranking function behind most keyword search engines.

- `query` is a list of words; `documents` is a list of documents, each a list of words.
- `N` is the number of documents, `avgdl` the average document length (in words), and for a word `q`, `n(q)` is the number of documents containing it.
- `idf(q) = ln((N - n(q) + 0.5) / (n(q) + 0.5) + 1)`
- For a document `D` with length `|D|`, where `f(q, D)` counts how often `q` appears in it:

  `score(D) = sum over the words q of the query of idf(q) * f(q, D) * (k1 + 1) / (f(q, D) + k1 * (1 - b + b * |D| / avgdl))`

- Go through the query words as given, so a word repeated in the query counts again.
- Return a NumPy array with one score per document.

In [20]:
def bm25_scores(query, documents, k1: float = 1.5, b: float = 0.75) -> np.ndarray:
    # your code here
    pass

In [21]:
check_bm25_scores(bm25_scores)

FAIL  bm25_scores: a one-word query
      expected: array(shape=(5,), dtype=float64, [0.7786, 0. , 0. , 1.7189, 0. ])
      got:      None


### Exercise 11: `LoRALinear`

Write a class `LoRALinear(base, r, alpha)`, an `nn.Module` that wraps an existing `nn.Linear` with a LoRA correction, as in Section 9 of the lesson.

- Keep the original layer as `self.base`, and freeze its weight and bias (`requires_grad = False`).
- Create `self.lora_A`, an `nn.Parameter` of shape `(r, in_features)`, initialised with `nn.init.kaiming_uniform_(..., a=math.sqrt(5))`, and `self.lora_B`, an `nn.Parameter` of shape `(out_features, r)` filled with **zeros**.
- Store `self.scaling = alpha / r`.
- `forward(x)` returns `self.base(x) + self.scaling * (x @ A^T @ B^T)`.
- Add a method `merged()` that returns a **new** `nn.Linear` whose weight is `W + scaling * B @ A` and whose bias is the base layer's bias (or no bias, if the base layer has none). It must give the same outputs as the LoRA layer.

In [22]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        # your code here

    def forward(self, x):
        # your code here
        pass

    def merged(self) -> nn.Linear:
        # your code here
        pass

In [23]:
check_LoRALinear(LoRALinear)

FAIL  LoRALinear: starts out identical to the base layer
      expected: tensor(shape=(5, 4), dtype=float32, [[ 0.1186, 0.1374, 0.472 , -0.6567], [-0.3746, -0.5...
      got:      None
      hint: B starts at zero


### Exercise 12: `quantize_int8`

Write `quantize_int8(weight)` that compresses a weight matrix to 8-bit integers, one of the ways large models are made to fit in less memory (Section 11 of the lesson).

Use **symmetric, per-row** quantisation:

- For each row, `scale = max(|row|) / 127`. If a row is all zeros, use `scale = 1.0`.
- `q = round(weight / scale)`, clipped to the range -127 to 127, stored as `torch.int8`.
- Return `(q, scales)`, where `scales` is a `float32` tensor of shape `(rows, 1)`.

The weights are approximately recovered as `q.float() * scales`. Every recovered value should be within `scale / 2` of the original.

Example: for the single row `[0.5, -1.27, 0.0]`, the scale is `0.01` and `q` is `[50, -127, 0]`.

In [24]:
def quantize_int8(weight: torch.Tensor):
    # your code here
    pass

In [25]:
check_quantize_int8(quantize_int8)

FAIL  quantize_int8: the example
      raised TypeError: 'NoneType' object is not subscriptable
      expected: [[50, -127, 0], [[0.01]]]


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, where you'll build and evaluate an assistant that answers questions about this course.